In [73]:

# === Whitelist ===
WHITELIST = {
    "intro", "outro", "interlinkages", "operationalize", "transformative", "underserved",
    "electrification", "blockchain", "intersectional", "interoperability", "decarbonization",
    "resilient", "localization", "digitization", "transactive", "unbanked", "gendered",
    "agrivoltaics", "agro", "aluminium", "analytics", "anonymization", "autoencoders",
    "backcasting", "bankability", "baseload", "behaviour", "bio", "bioclimatic",
    "bioenergy", "bioethanol", "biofuels", "centres", "cleantech", "counterparites",
    "crowdfunding", "cyberattacks", "cybersecurity", "dataset", "datasets",
    "digitalization", "disincentivizing", "dispatchable", "ecookbook", "endeavour",
    "endeavours", "etc", "favourable", "fuelwood", "funders", "geospatial",
    "greenwashing", "hexafluoride", "hoc", "hypothetication", "impactful",
    "incentivizing", "inclusivity", "interconnectivity", "intergenerational",
    "intersectionality", "intertemporal", "investable", "issuances", "kwh", "labour",
    "levelized", "lifecycle", "metadata", "microenterprises", "microfinance",
    "microgrid", "microgrids", "minigrid", "minigrids", "multi", "overconsumption",
    "perovskite", "photovoltaics", "pre", "programme", "programmes", "prosumers",
    "reimagining", "renewables", "repurposing", "reputational", "reskilling",
    "roadmap", "roadmaps", "securitization", "servitization", "smartphones", "socio",
    "stressors", "subnational", "subsector", "superbond", "tech", "terawatt",
    "timeframe", "timelines", "underrepresentation", "underutilization",
    "unelectrified", "unserved", "upskilling", "wastewater","sukuk","agri", "agroforestry", "agrofuels", "analyse", "analysed", "answerability",
"approx", "auditable", "autothermal", "behavioural", "biofuel", "biomethane",
"biopower", "catalyse", "centre", "centred", "characterised", "characterising",
"chatbots", "chokepoints","microloans","app", "counterparty", "crowdfunded", "cryptocurrency", "cyber", "decisionmakers", "degrowth", "deliverables", "derisking", "disruptors", "electrolyzer", "electromobility", "embeddedness", "enablement", "etf", "extractives", "favela", "feebates", "financeable", "fintech", "fracking", "frontlines", "fundable", "gasification", "gigatonnes", "governorates", "graphene", "greenwashed", "hyperparameters", "hypothecation",
    "reskill", "exajoules", "supercapacitors", "prosumer", "onsite", "pico",
"tarifa", "energia", "intra", "reframes", "terawatts", "monocrystalline", "diselenide",
"kg", "rebalancing", "cybercriminal", "organisations", "siloed", "underrepresent",
"underbanked", "todo", "rebalance", "lightbulbs", "megatonnes", "reimagine",
"overexploitation", "monocropping", "utilise", "multidimensionally", "replenishable",
"org", "outcompeting", "microclimates", "webinars", "ramping", "decarbonise",
"wellbeing", "mortalities", "maladaptation", "derated", "amunas", "unmanaged",
"ie", "adjuntas", "eq", "subsea", "transboundary", "overaccumulation", "pastoralists",
"incentivizes", "upskilled", "skillset", "subprocesses", "uptime", "malware",
"middleware", "ontologies", "operationalization", "workflow", "underrepresenting",
"tuk", "tuks", "overfitting", "privacies", "ii", "iii", "legislations", "underfitting",
"racialized", "situ", "microloan", "preprocessing", "learnings", "onboarded", "dejan",
"fueron", "muertos", "incendios", "destructivos", "los", "costos", "chileno", "carbono",
"neutralidad", "requerimientos", "financieros", "kilometre", "incentivized",
"megatrends", "webinar", "overregulating", "reactively", "toolkits", "cryptocurrencies",
"timeframes", "trialling", "whistleblowers", "tri", "recognise", "quo", "underpriced",
"operationalizing", "todos", "telemedicine", "screenshot", "decentralised", "infographics",
"specificities", "misallocating", "km", "tokenized", "microfinancing", "organised",
"realise", "recognising", "upskill", "peatlands", "transdisciplinary", "vis", "multisectoral",
"repurpose", "tokenistic", "marginalised", "synergizing", "oversaturation", "rollout",
"extractivist", "marketization", "renegotiations", "unsustainability", "tradability",
"decarbonised", "underperformance", "multilaterals", "multistakeholder", "counterparties",
"kwp", "steelmaking", "securitising", "securitisation", "undiversified", "prosumerism",
"interdependencies","por", "seforall","un","iea"

}


In [74]:
import os
import json
import re
import hashlib
import pandas as pd
from pathlib import Path
from spellchecker import SpellChecker
import html

# === Paths ===
LESSON_FOLDER = "../03_Outputs/SEA_Modules/en"
OUTPUT_CSV = "../03_Outputs/spellcheck_audit_report.csv"

# === Setup spellchecker ===
spell = SpellChecker(language="en")
spell.word_frequency.load_words([
    "UNDP", "AI", "SDG", "SEA", "ImageKit", "Figma", "infographic",
    "CTA", "pdf", "webp", "photobank", "electrification", "climate", "energy", "Africa"
])
# Load any special words you have elsewhere here, e.g. from your whitelist file
# For example: spell.word_frequency.load_words(my_whitelist_words)

# === Helpers ===
def remove_html_tags(text: str) -> str:
    text = re.sub(r'<br\s*/?>', '. ', text)  # replace <br> or <br/> with period+space
    text = re.sub(r'<[^>]+>', '', text)     # remove other tags
    return text

def is_mixed_case(word: str):
    """Returns True if the word is neither all lowercase nor all uppercase"""
    return not (word.isupper() or word[0].isupper())

def should_check(word: str):
    """Return True if the word should be spellchecked"""
    return (
        word.lower() not in WHITELIST
        and word.isalpha()
        and is_mixed_case(word)
    )

def split_text_into_sentences(text: str):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

import re
URL_PATTERN = re.compile(
    r"(https?://|www\.|[a-zA-Z0-9\-]+\.[a-zA-Z]{2,}|(?:\w+/){2,}\w+)"
)

def clean_mojibake(text: str) -> str:
    if not isinstance(text, str):
        return text
    replacements = {
        "â€™": "’", "â€˜": "‘", "â€œ": "“", "â€": "”",
        "â€“": "–", "â€”": "—", "â€¦": "…",
        "Ã©": "é", "Ã¨": "è", "Ã": "à",
        "‚Äô": "’", "‚Äì": "–", "‚Äî": "—", "‚Äú": "“", "‚Äù": "”",
        "Â ": "", "Â": ""
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    text = html.unescape(text)
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r'([.,!?])(?=[^\s])', r'\1 ', text)
    return text

# === Sentence extraction functions ===

def extract_sentences_with_frame_count(lesson_json, lesson_id=None):
    """
    Extracts spellcheckable sentences from lesson JSON.
    Each 'frame' refers to a segment (an item in the 'segments' list).
    """
    records = []
    segments = lesson_json.get("segments", [])

    for frame_index, segment in enumerate(segments, start=1):
        def walk(obj, path):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if isinstance(v, str) and (
                        k in {"label", "title", "intro", "text", "body", "description", "cta", "value", "prompt"}
                        or (path and path[-1] == "labels")
                    ):
                        clean_text = clean_mojibake(remove_html_tags(v))
                        sentences = split_text_into_sentences(clean_text)
                        for sentence in sentences:
                            if URL_PATTERN.search(sentence.lower()):
                                continue
                            records.append({
                                "lesson_id": lesson_id,
                                "frame": frame_index,
                                "json_path": path + [k],
                                "sentence_text": sentence.strip()
                            })
                    else:
                        walk(v, path + [k])
            elif isinstance(obj, list):
                for idx, item in enumerate(obj):
                    walk(item, path + [idx])
        
        walk(segment, path=[frame_index])

    return records


def extract_sentences_old(lesson_json):
    """
    Old extraction without frame counting, used for module_structure.json or special files.
    Each sub-object should contain an "id" that will be used as the lesson_id.
    """
    records = []

    def walk(obj, path, current_lesson_id=None):
        if isinstance(obj, dict):
            # Update lesson_id if this object contains it
            lesson_id = obj.get("id", current_lesson_id)

            for k, v in obj.items():
                if isinstance(v, str) and (
                    k in {"label", "title", "intro", "text", "body", "description", "cta", "value", "prompt"}
                    or (path and path[-1] == "labels")
                ):
                    clean_text = clean_mojibake(remove_html_tags(v))
                    sentences = split_text_into_sentences(clean_text)
                    for sentence in sentences:
                        if URL_PATTERN.search(sentence.lower()):
                            continue
                        records.append({
                            "lesson_id": "ToC",
                            "frame": lesson_id,  # Use lesson_id as frame for module_structure
                            "json_path": path + [k],
                            "sentence_text": sentence.strip()
                        })
                else:
                    walk(v, path + [k], lesson_id)
        elif isinstance(obj, list):
            for idx, item in enumerate(obj):
                walk(item, path + [idx], current_lesson_id)

    walk(lesson_json, [])
    return records




# === Spellcheck single file ===
def audit_file(path):
    issues = []
    try:
        with open(path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)
    except Exception as e:
        print(f"❌ Failed to load {path}: {e}")
        return issues

    lesson_id = data.get("id", Path(path).stem)

    # Special case: if filename is module_structure.json, use old extraction (no frame)
    if Path(path).name == "module_structure.json":
        sentences = extract_sentences_old(data)
    else:
        sentences = extract_sentences_with_frame_count(data, lesson_id=lesson_id)

    for item in sentences:
        sentence = item["sentence_text"]
        words = re.findall(r"\b[a-zA-Z]+'?[a-zA-Z]+\b", sentence)
        filtered = [w for w in words if should_check(w)]
        misspelled = spell.unknown(filtered)
        for word in misspelled:
            issues.append({
                "lesson_id": item["lesson_id"],
                "frame": item["frame"],
                "misspelled_word": word,
                "sentence": sentence
            })
    return issues

# === Process all files ===
def audit_all_lessons(folder):
    all_issues = []
    for path in Path(folder).rglob("*.json"):
        issues = audit_file(path)
        all_issues.extend(issues)
    return pd.DataFrame(all_issues)

# === Run audit ===
if __name__ == "__main__":
    df_issues = audit_all_lessons(LESSON_FOLDER)
    df_issues.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Spellcheck complete — {len(df_issues)} issues found across {df_issues['lesson_id'].nunique()} lessons")
    print(f"📄 Report saved to: {OUTPUT_CSV}")


✅ Spellcheck complete — 75 issues found across 42 lessons
📄 Report saved to: ../03_Outputs/spellcheck_audit_report.csv
